## Module 5 Class activities
This notebook is a starting point for the exercises and activities that we'll do in class. We'll do an extension of the random forests classifier, looking at a continuous variable.

Before you attempt any of these activities, make sure to watch the video lectures for this module.

### Classification: NYC evictions
We'll look at the factors that are associated with evictions in New York City. Perhaps a machine learning model can identify the types of places that are vulnerable to eviction, and target renter assistance programs more effectively?

#### Loading in the data

Let's start by loading in the [eviction dataset](https://data.cityofnewyork.us/City-Government/Evictions/6z8x-wfk4) via Socrata.

<div class="alert alert-block alert-info">

<strong>Exercise:</strong> Import the data from Socrata via the API into a pandas DataFrame.
</div>

*Hints*:
- Look back at Week 1 if you need a refresher on using Socrata
- There are about 70,000 rows in the dataset. So remember to add `?$limit=100000` to the end of the URL that you pass to `requests.get()`. Otherwise, you'll just get the first 1,000 rows. (The limit can be anything comfortably above 70000.)

In [1]:
import requests
import json
import pandas as pd
import geopandas as gpd

# your code here
url = 'https://data.cityofnewyork.us/resource/6z8x-wfk4.json?$limit=100000'
r = requests.get(url)
evictionDf = pd.DataFrame(json.loads(r.text)) 
evictionDf.head()

,court_index_number,docket_number,eviction_address,executed_date,marshal_first_name,marshal_last_name,residential_commercial_ind,borough,eviction_zip,ejectment,eviction_possession,eviction_apt_num,latitude,longitude,community_board,council_district,census_tract,bin,bbl,nta
0,53609/17,007357,221-16 69TH AVENUE ALL FLOORS,2017-06-01T00:00:00.000,George,"Essock, Jr.",Residential,QUEENS,11364,Not an Ejectment,Possession,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,305759/20B,120693,532 WILLIAMS AVENUE,2022-11-30T00:00:00.000,Darlene,Barone,Residential,BROOKLYN,11207,Not an Ejectment,Possession,4B,40.662938,-73.897446,5,42,1130,3084965,3038180052,East New York (Pennsylvania Ave)
2,73308/18,20526,190-20 99TH AVENUE,2019-07-15T00:00:00.000,Edward,Guida,Residential,QUEENS,11423,Not an Ejectment,Possession,2F,40.709690,-73.767545,12,27,50201,4310708,4108390019,Hollis
3,82506/18,088432,425 NEPTUNE AVENUE,2019-01-28T00:00:00.000,Henry,Daley,Residential,BROOKLYN,11224,Not an Ejectment,Possession,16C,40.580037,-73.969407,13,47,35601,3320734,3072530001,West Brighton
4,320292/23,132420,2546 ADAM CLAYTON PO WELL JR. BOULEVARD,2024-09-19T00:00:00.000,Justin,Grossman,Commercial,MANHATTAN,10039,Not an Ejectment,Possession,"STORE #2,COMMERCIAL",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<div class="alert alert-block alert-info">

<strong>Exercise:</strong> Convert your dataframe to a GeoDataFrame, using the latitude and longitude columns.

In [2]:
# your code here 

evictionGdf = gpd.GeoDataFrame(
    evictionDf, geometry=gpd.points_from_xy(evictionDf.longitude, evictionDf.latitude, 
                                          crs='EPSG:4326'))
evictionGdf

,court_index_number,docket_number,eviction_address,executed_date,marshal_first_name,marshal_last_name,residential_commercial_ind,borough,eviction_zip,ejectment,...,eviction_apt_num,latitude,longitude,community_board,council_district,census_tract,bin,bbl,nta,geometry
0,53609/17,007357,221-16 69TH AVENUE ALL FLOORS,2017-06-01T00:00:00.000,George,"Essock, Jr.",Residential,QUEENS,11364,Not an Ejectment,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY
1,305759/20B,120693,532 WILLIAMS AVENUE,2022-11-30T00:00:00.000,Darlene,Barone,Residential,BROOKLYN,11207,Not an Ejectment,...,4B,40.662938,-73.897446,5,42,1130,3084965,3038180052,East New York (Pennsylvania Ave),POINT (-73.89745 40.66294)
2,73308/18,20526,190-20 99TH AVENUE,2019-07-15T00:00:00.000,Edward,Guida,Residential,QUEENS,11423,Not an Ejectment,...,2F,40.709690,-73.767545,12,27,50201,4310708,4108390019,Hollis,POINT (-73.76754 40.70969)
3,82506/18,088432,425 NEPTUNE AVENUE,2019-01-28T00:00:00.000,Henry,Daley,Residential,BROOKLYN,11224,Not an Ejectment,...,16C,40.580037,-73.969407,13,47,35601,3320734,3072530001,West Brighton,POINT (-73.96941 40.58004)
4,320292/23,132420,2546 ADAM CLAYTON PO WELL JR. BOULEVARD,2024-09-19T00:00:00.000,Justin,Grossman,Commercial,MANHATTAN,10039,Not an Ejectment,...,"STORE #2,COMMERCIAL",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT EMPTY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,48679/19,27387,1466 HICKS STREET,2023-09-18T00:00:00.000,Edward,Guida,Residential,BRONX,10469,Not an Ejectment,...,NaN,40.878931,-73.848805,12,12,386,2060203,2047210109,Eastchester-Edenwald-Baychester,POINT (-73.8488 40.87893)
99996,318963/22,107545,1935 ANDREWS AVENUE,2023-01-04T00:00:00.000,Henry,Daley,Residential,BRONX,10453,Not an Ejectment,...,5D,40.854764,-73.913171,5,14,24502,2014888,2032210074,University Heights-Morris Heights,POINT (-73.91317 40.85476)
99997,300558/20,118681,1385 NELSON AVENUE,2023-08-03T00:00:00.000,Justin,Grossman,Residential,BRONX,10452,Not an Ejectment,...,3F,40.841487,-73.922694,4,16,211,2003348,2025210027,Highbridge,POINT (-73.92269 40.84149)
99998,307246/23,124794,964 DEKALB AVENUE,2025-02-14T00:00:00.000,Justin,Grossman,Commercial,BROOKLYN,11221,Not an Ejectment,...,STORE FRONT,40.693038,-73.936829,3,36,289,3043243,3016020010,Stuyvesant Heights,POINT (-73.93683 40.69304)


Now let's import some census data. We could use `cenpy` or the Census Bureau API. But to keep things simple so that we can focus on the spatial joins and the machine learning, I downloaded the block group-level 2019 ACS data for New York from the [Census Bureau](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-data.html). To save space, I clipped it to the 5 NYC counties.

It's in your repository, and we can load it in as follows. If you aren't familiar with a GeoPackage (GPKG) format, think of it as a "new and improved shapefile." [Here's a good overview.](https://towardsdatascience.com/why-you-need-to-use-geopackage-files-instead-of-shapefile-or-geojson-7cb24fe56416)

In [3]:
bgs = gpd.read_file('data/nyc_bgs.gpkg')
bgs.head()

,GEOID,B01001e1,B01001e10,B01001e11,B01001e12,B01001e13,B01001e14,B01001e15,B01001e16,B01001e17,...,B19001e8,B19001e9,B22010e1,B22010e2,B22010e3,B22010e4,B22010e5,B22010e6,B22010e7,geometry
0,15000US360050175002,656,39,0,0,0,18,14,0,22,...,11,0,358,214,139,75,144,107,37,"POLYGON ((-73.9157 40.83054, -73.91485 40.8302..."
1,15000US360050141001,1228,0,35,96,26,45,28,0,54,...,34,0,503,291,226,65,212,70,142,"POLYGON ((-73.91661 40.82499, -73.91592 40.825..."
2,15000US360050145001,2716,44,192,33,38,76,30,64,93,...,137,83,972,534,316,218,438,7,431,"POLYGON ((-73.90584 40.83106, -73.90505 40.832..."
3,15000US360050075002,3488,43,109,122,169,19,51,139,22,...,63,37,1188,470,300,170,718,147,571,"POLYGON ((-73.91035 40.81995, -73.91022 40.820..."
4,15000US360050418001,657,0,38,5,11,21,14,43,25,...,0,0,217,87,18,69,130,41,89,"POLYGON ((-73.86288 40.89515, -73.86146 40.897..."


Note that the variables aren't particularly carefully selected - I just threw in many of the demographic and housing variables. 

Nor are the variable names particularly informative, but the full names are in a file in the repository.

In [4]:
# note it is tab-sepated, not comma separated
# so we use the sep='\t' argument

col_names = pd.read_csv('data/BG_METADATA_2019.txt', sep='\t', index_col='Short_Name')
col_names.head()

,Full_Name
Short_Name,
B01001e1,SEX BY AGE: Total: Total population -- (Estimate)
B01001m1,SEX BY AGE: Total: Total population -- (Margin...
B01001e2,SEX BY AGE: Male: Total population -- (Estimate)
B01001m2,SEX BY AGE: Male: Total population -- (Margin ...
B01001e3,SEX BY AGE: Male: Under 5 years: Total populat...


So you can see the definition of the column like this. (I don't recommend renaming the `bg` column names, because the full names are so long.)

In [5]:
col_names.loc['B01001e1']

Full_Name    SEX BY AGE: Total: Total population -- (Estimate)
Name: B01001e1, dtype: object

#### Spatial join
Now let's do the spatial join. Again, let's follow our three step process.

1. Use a spatial join to add the `GEOID` column to the evictions dataframe. *Hint:* Check your projections.
2. Group by `GEOID` to get a count of evictions per block group. If you have a `Series`, give it a name - maybe `n_evictions`
3. Join those counts back - a tabular join based on the index

<div class="alert alert-block alert-info">
    <strong>Exercise:</strong> Add a count of evictions per census block group to your <strong>bgs</strong> GeoDataFrame, using the 3-step process above.
</div>

In [6]:
print(evictionGdf.crs)
print(bgs.crs)

EPSG:4326
EPSG:4269


In [7]:
evictionGdf_sjoin = gpd.sjoin(evictionGdf, bgs.to_crs('EPSG:4326'), predicate='intersects')
print(len(evictionGdf))
print(len(bgs))
print(len(evictionGdf_sjoin))

100000
6493
91036


In [8]:
evictionGdf_sjoin.groupby('GEOID').size()

GEOID
15000US360050002001    10
15000US360050002002    17
15000US360050002003     9
15000US360050004001    14
15000US360050004002    17
                       ..
15000US360850319012    13
15000US360850319021    82
15000US360850319022    40
15000US360850319023    33
15000US360850323001    23
Length: 5921, dtype: int64

In [9]:
temp_evictionDf = evictionGdf_sjoin.groupby('GEOID').size()
temp_evictionDf = pd.DataFrame(temp_evictionDf)
temp_evictionDf.columns = ['n_evictions']

In [10]:
temp_evictionDf

,n_evictions
GEOID,
15000US360050002001,10
15000US360050002002,17
15000US360050002003,9
15000US360050004001,14
15000US360050004002,17
...,...
15000US360850319012,13
15000US360850319021,82
15000US360850319022,40


In [11]:
evictionGdf_sjoin.set_index('GEOID', inplace=True) 

In [12]:
evictionGdf_sjoin

,court_index_number,docket_number,eviction_address,executed_date,marshal_first_name,marshal_last_name,residential_commercial_ind,borough,eviction_zip,ejectment,...,B19001e7,B19001e8,B19001e9,B22010e1,B22010e2,B22010e3,B22010e4,B22010e5,B22010e6,B22010e7
GEOID,,,,,,,,,,,,,,,,,,,,,
15000US360471130002,305759/20B,120693,532 WILLIAMS AVENUE,2022-11-30T00:00:00.000,Darlene,Barone,Residential,BROOKLYN,11207,Not an Ejectment,...,27,27,11,491,59,20,39,432,129,303
15000US360810502011,73308/18,20526,190-20 99TH AVENUE,2019-07-15T00:00:00.000,Edward,Guida,Residential,QUEENS,11423,Not an Ejectment,...,25,41,0,372,38,17,21,334,49,285
15000US360470356011,82506/18,088432,425 NEPTUNE AVENUE,2019-01-28T00:00:00.000,Henry,Daley,Residential,BROOKLYN,11224,Not an Ejectment,...,77,25,45,1482,531,310,221,951,265,686
15000US360050369022,23469/17,471256,1785 PROSPECT AVENUE,2017-09-20T00:00:00.000,Danny,Weinheim,Residential,BRONX,10457,Not an Ejectment,...,0,0,7,526,348,175,173,178,34,144
15000US360050302005,15179/17,167890,100 ELGAR PLACE,2017-09-22T00:00:00.000,Alfred,Locascio,Residential,BRONX,10475,Not an Ejectment,...,182,70,105,2026,107,0,107,1919,412,1507
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15000US360050386008,48679/19,27387,1466 HICKS STREET,2023-09-18T00:00:00.000,Edward,Guida,Residential,BRONX,10469,Not an Ejectment,...,90,25,0,287,25,25,0,262,20,242
15000US360050245023,318963/22,107545,1935 ANDREWS AVENUE,2023-01-04T00:00:00.000,Henry,Daley,Residential,BRONX,10453,Not an Ejectment,...,17,18,55,631,400,205,195,231,78,153
15000US360050211002,300558/20,118681,1385 NELSON AVENUE,2023-08-03T00:00:00.000,Justin,Grossman,Residential,BRONX,10452,Not an Ejectment,...,35,42,24,547,293,85,208,254,55,199


In [13]:
bgs.set_index('GEOID', inplace=True) 

In [14]:
evictionGdf_sjoin = temp_evictionDf.join(evictionGdf_sjoin, how='left')

In [15]:
evictionGdf_sjoin

,n_evictions,court_index_number,docket_number,eviction_address,executed_date,marshal_first_name,marshal_last_name,residential_commercial_ind,borough,eviction_zip,...,B19001e7,B19001e8,B19001e9,B22010e1,B22010e2,B22010e3,B22010e4,B22010e5,B22010e6,B22010e7
GEOID,,,,,,,,,,,,,,,,,,,,,
15000US360050002001,10,040450/16,071109,420 THIERIOT AVENUE,2017-06-16T00:00:00.000,Henry,Daley,Residential,BRONX,10473,...,12,0,70,460,154,55,99,306,109,197
15000US360050002001,10,51142/18,171680,442 ST. LAWRENCE AVE,2019-06-03T00:00:00.000,Alfred,Locascio,Residential,BRONX,10473,...,12,0,70,460,154,55,99,306,109,197
15000US360050002001,10,B26692/19,099260,1793 PATTERSON AVENU E,2019-09-17T00:00:00.000,Ileana,Rivera,Residential,BRONX,10473,...,12,0,70,460,154,55,99,306,109,197
15000US360050002001,10,B34874/17,099932,455 BEACH AVENUE,2017-11-06T00:00:00.000,Darlene,Barone,Residential,BRONX,10473,...,12,0,70,460,154,55,99,306,109,197
15000US360050002001,10,309311/22,110105,422 LELAND AVENUE,2023-01-18T00:00:00.000,Justin,Grossman,Residential,BRONX,10473,...,12,0,70,460,154,55,99,306,109,197
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15000US360850323001,23,R 52317/16,068284,158 HOLLAND AVENUE,2017-09-20T00:00:00.000,Steven,Powell,Residential,STATEN ISLAND,10303,...,0,9,29,451,79,19,60,372,25,347
15000US360850323001,23,50748/18,006138,54 HOLLAND AVENUE,2018-12-03T00:00:00.000,Frank,Siracusa,Residential,STATEN ISLAND,10303,...,0,9,29,451,79,19,60,372,25,347
15000US360850323001,23,R50972/18,090889,84 HOLLAND AVENUE,2018-11-08T00:00:00.000,Ileana,Rivera,Residential,STATEN ISLAND,10303,...,0,9,29,451,79,19,60,372,25,347


In [16]:
# your code here



<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Do a quick-and-dirty map of the number of evictions. This will help identify any data holes.
</div>

In [17]:
# your code here

#### Random forests regressor
Now we have our data set. Let's estimate a random forests model.

In contrast to the examples in the lecture, we are trying to predict a continuous variable - the number of evictions. So our classifier isn't appropriate. 

However, there is a similar model: the [random forest regressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor). It works almost identically to the classifier. The main difference from a user perspective is assessing model performance - a confusion matrix doesn't work here.

You'll need to follow the following steps:
- choose your x variables. (Your y variable will be `n_evictions`)
- Drop Null values if needed
- split your dataset into training and testing portions
- estimate (fit) the model

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Estimate a random forest regressor model to predict the number of evictions per census tract.</div>

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# your code here

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Examine some of your trees in the random forest. What do they tell you?</div>

In [19]:
# your code here

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Experiment with different model hyperparameters and variables. Discuss your rationale and the results with a neighbor.</div>

In [20]:
# your code here

The following questions relate to some of the material in Module 6. You might want to wait until watching those lectures. Then come back and complete these tasks.

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Assess the fit of your model.</div>

Remember, the confusion matrix and accuracy scores don't apply to continuous data. Some ideas for continuous variables are [here](https://stackoverflow.com/questions/50789508/random-forest-regression-how-do-i-analyse-its-performance-python-sklearn). You could also plot actual vs predicted values.

In [21]:
# your code here

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Which variables are most important in your predictions? Plot the forest importances.</div>

In [22]:
# your code here


<div class="alert alert-block alert-info">
<h3>What you should have learned</h3>
<ul>
  <li>Get more practice with spatial joins and Socrata.</li>
  <li>Learn how to estimate a random forests model for continuous data.</li>
</ul>
</div>